<img src="logo.png" alt="Vegeta" width="240">

# NISUS+ — the mountain survey drone (33)

*Formerly FALCO: the name FALCO now belongs to the tractor redesign; this twin-boom pusher is NISUS's bigger
mountain sibling, NISUS+, in one variant, Nisus+ Zero.*

**Nisus+ Zero** is NISUS's style made bigger and given the energy and the power for the mountains: a 2.4 m twin-boom
single pusher with an H-tail and a rounded pod, an NVIDIA Jetson Orin onboard for autonomous survey and computer vision,
a 6S Li-ion pack of 21700 cells, a 1.5 kW drive. It is built to **climb fast** (≥ 8 m/s at sea level, ≥ 5 m/s at
4500 m: enough to out-climb a lee downdraft), to **come down fast** (≥ 10 m/s vertical with **crow** — the flaps
down, the ailerons up — and the **propeller brake**, which also regenerates), to work **up to 4500 m**, and to go up
the mountain in a backpack (a three-piece wing).

This notebook is the **first engineering approximation**, as NISUS's (notebook 31): every number is calculated, sourced
or labelled as an assumption; missing analyses say **NOT RUN**. What is new against NISUS: the air thins with height
(everything takes the density), the drive keeps its sign (the windmill brake and the regeneration), the pack sags and
gets cold, the flaps and crow, the mountain terrain and wind in MuJoCo, and a bird hunt over the slopes.

| part | what | where the code is |
|---|---|---|
| 1 | requirements and the starting shape | `designs/nisus_plus.py` |
| 2 | CAD (Dedalus): parts, three-view, crow, transport, exploded view, propeller disk, packaging | `nisus_plus.py`, `nisus_plus_drawings.py` |
| 3 | materials, construction, load paths | `nisus_plus_systems.py`, `nisus_plus_structure.py` |
| 4 | the atmosphere; mass, CG, inertia, wing loading | `nisus_plus_systems.py` |
| 5 | electronics and their energy | `nisus_plus_systems.py` |
| 6 | propulsion: requirement at altitude, selection, the drive over altitude, the brake region, the Li-ion pack | `nisus_plus_systems.py` |
| 7 | aerodynamics: lattice with flaps, crow, derivatives, CFD at altitude (Aeromant / OpenFOAM) | `nisus_plus_flight.py`, `nisus_plus_cfd.py` |
| 8 | structure: load cases in EAS, hand checks, FEA (Talos / CalculiX) | `nisus_plus_structure.py` |
| 9 | flight: trim, CG range, the envelope against altitude, climb strategy, descent, regeneration, launch, limits | `nisus_plus_flight.py` |
| 10 | mission energy over the mountains, the battery iteration | `nisus_plus_systems.py` |
| 11 | MuJoCo (Chiron) missions over the mountains, with movies; 11b the bird hunt | `nisus_plus_robot.py`, `nisus_plus_controller.py`, `nisus_plus_scenario.py`, `nisus_plus_birds.py` |
| 12 | does regeneration pay? the first recommended build; what to refine next | — |

Switches: `VEGETA_DEBUG=1` mocks FEA and CFD; `NISUS_PLUS_QUICK=1` flies only the calm mission and one bird hunt, small
movies; `VEGETA_SKIP_OPENFOAM=1` skips the CFD (NOT RUN); `NISUS_PLUS_FEA=0` skips the FEA (NOT RUN).

In [ ]:
# DEBUG: test the notebook end to end before the full run (FEA and CFD mocked). On here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute 33_nisus_plus.ipynb --output 33_nisus_plus-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Video, display
from vegeta import dedalus, talos, aeromant, boreas, cache
sys.path.insert(0, "designs")
import nisus_plus, nisus_plus_systems as fs, nisus_plus_flight as ff, nisus_plus_structure as fst, nisus_plus_cfd as fcfd, nisus_plus_drawings as fd
import nisus_plus_robot as fr, nisus_plus_controller as fc, nisus_plus_scenario as fsc, nisus_plus_birds as fb

pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
ROOT = Path("_runs/nisus_plus"); ROOT.mkdir(parents=True, exist_ok=True)
OUT = Path("output") / "33_nisus_plus"; OUT.mkdir(parents=True, exist_ok=True)
for sub in ("cad", "drawings", "tables", "telemetry", "plots"):
    (OUT / sub).mkdir(exist_ok=True)
cache.notebook("33_nisus_plus")
QUICK = os.environ.get("NISUS_PLUS_QUICK") == "1"
RUN_FEA = os.environ.get("NISUS_PLUS_FEA", "1") != "0"
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
G, RHO0 = fs.G, fs.RHO0
STATUS = {}
print(f"DEBUG {DEBUG} | QUICK {QUICK} | FEA {'on' if RUN_FEA else 'NOT RUN'} | CFD {'on' if RUN_CFD else 'NOT RUN'}")
def save_table(df, name):
    df.to_csv(OUT / "tables" / f"{name}.csv")
    return df

# Part 1 — requirements and the starting shape

| requirement | target | why |
|---|---|---|
| operating altitude | launch sites up to 3000 m, work up to 4500 m (density altitude) | the Alps; a hot day raises the density altitude |
| climb | ≥ 8 m/s at sea level, **≥ 5 m/s at 4500 m** (MTOW) | out-climb a 4 m/s lee downdraft with 1 m/s to spare |
| descent | **≥ 10 m/s** vertical at ≤ V_FE, speed held | escape weather; get out of a ridge lift that carries the aircraft to the cloud; a steep approach into a meadow |
| energy | a 20-min terrain-following survey after a 2800 m climb, 20 % reserve | the mission of Part 10 |
| wind | 12 m/s mean, 8-10 m/s gusts, ±4 m/s slope lift / lee sink | mountain air |
| transport | longest piece ≤ ~1.1 m | strapped to a backpack |
| computer | one variant, **Nisus+ Zero**: the Jetson Orin Nano Super (the Nano B01 is end-of-life) | autonomy, the bird mission |

The starting shape is NISUS's (notebook 31) scaled: span 2.4 m, chords 300/200 mm (S = 0.60 m², AR 9.6: at altitude
the induced drag pushes toward a higher aspect ratio), the pod for a 6S pack and the Orin, booms 500 mm apart, a
15-inch pusher. New: inboard flaps and outboard ailerons (crow), the panel joint with a spar joiner. The first
structural checks (Part 8) changed the spar to 20/17 mm, the rear tube to 18/14 mm and the booms to 20/18 mm, and
lengthened the boom socket.

In [ ]:
P = nisus_plus.resolve()
L = nisus_plus.NisusPlus.layout(P)
DES = nisus_plus.NisusPlus()
pt = pd.DataFrame([{"parameter": q.name, "value": P[q.name], "unit": q.units, "description": q.description} for q in nisus_plus.NisusPlus.parameters if q.name in P]).set_index("parameter")
display(save_table(pt, "parameters"))
display(pd.Series({k: L[k] for k in ("S_ref", "AR", "mac", "V_h", "V_v", "l_t", "S_flap_each", "S_aileron_each", "prop_clearance_boom", "prop_ground_margin_resting",
                                     "overall_length", "longest_piece_mm")}, name="layout").to_frame())

# Part 2 — CAD (Dedalus)

In [ ]:
t0 = time.time()
GEO = DES.generate()
print(GEO.measure())
for part in ("aircraft", "wing", "pod", "tail", "boom_fitting", "motor_mount", "spar_joiner", "flap", "aileron"):
    g = DES.generate(part=part)
    g.export_step(str(OUT / "cad" / f"nisus_plus_{part}.step"))
GEO.export_stl(str(OUT / "cad" / "nisus_plus_aircraft.stl"), tolerance=0.3)
CROW_GEO = DES.generate(flap_deg=55.0, aileron_deg=-25.0)
CROW_GEO.export_stl(str(OUT / "cad" / "nisus_plus_crow.stl"), tolerance=0.3)
print("crow pose:", CROW_GEO.measure()["valid"], CROW_GEO.measure()["n_solids"], "solid(s);", f"{time.time() - t0:.0f} s")
fig = fd.three_view(P); fig.savefig(OUT / "drawings" / "three_view.png", dpi=110); plt.show()

In [ ]:
fig = fd.crow_view(P); fig.savefig(OUT / "drawings" / "crow.png", dpi=110); plt.show()
fig = fd.transport_view(P); fig.savefig(OUT / "drawings" / "transport.png", dpi=110); plt.show()
display(pd.Series(nisus_plus.transport_check(P)).to_frame("transport"))
try:
    fd.exploded_png(OUT / "drawings" / "exploded.png", P)
    display(Image(str(OUT / "drawings" / "exploded.png"), width=900))
except Exception as e:
    print("the exploded view needs pyvista off screen (xvfb-run):", e)

## The propeller's swept disk and the packaging

In [ ]:
PC, fig = fd.prop_clearance(P); fig.savefig(OUT / "drawings" / "prop_clearance.png", dpi=110); plt.show()
display(PC)
display(pd.Series(nisus_plus.boom_check(P)).drop("tube").to_frame("boom by hand"))
fig = fd.internal_layout(fs.DEFAULT_PACK, P); fig.savefig(OUT / "drawings" / "internal_layout.png", dpi=110); plt.show()

# Part 3 — materials, construction and load paths

NISUS's construction at the larger size: an XPS foam core (25 % lightened) with a **2 mm balsa D-box over the whole span**
(the bigger wing flies faster and twists more), a **20/17 mm carbon spar** in three pieces with **carbon joiners** at the
panel joints (the outer panels slide on and lock with a pin), an 18/14 mm rear carry-through tube, 20/18 mm carbon booms
in printed PETG root fittings that clamp both tubes, an H-tail of 11 mm foam plates, an LW-PLA printed pod, a TPU
belly skid. Load paths: the lift → the spar → the joiner → the pod's saddle; the tail's loads → the booms → the root
fittings → the spar and the rear tube as a couple; the thrust and the brake's reverse thrust → the motor mount → the pod.

In [ ]:
display(pd.DataFrame(fs.MATERIALS).T)
display(pd.DataFrame(fs.structure_items(P), columns=["item", "group", "mass [g]", "x [mm]", "y [mm]", "z [mm]", "basis"]).set_index("item").round(1))

# Part 4 — the air, the mass, the centre of gravity, the inertia

The ISA troposphere (`vegeta.boreas.microjet.isa` with a temperature offset) and Sutherland's viscosity: at 4500 m the
density is 63 % of sea level's and the kinematic viscosity 42 % higher (the Reynolds numbers fall). The mass table puts
the 6S pack along its bay under the wing so the CG sits at 28 % MAC.

In [ ]:
display(save_table(fs.atmosphere_table(), "atmosphere"))
display(fs.atmosphere_table(dT=20.0)[["ρ [kg/m³]", "density altitude [m]"]].rename(columns=lambda c: c + " (ISA +20)"))
MT = {k: fs.mass_table(k, p=P) for k in fs.PACKS}
CI = {k: fs.cg_inertia(t, P) for k, t in MT.items()}
display(save_table(MT[fs.DEFAULT_PACK].round(1), "mass_table"))
display(MT[fs.DEFAULT_PACK].groupby("group")["mass [g]"].sum().round(0).to_frame())
display(pd.DataFrame({k: {x: c[x] for x in ("mass_kg", "x_cg_frac_mac", "Ixx", "Iyy", "Izz", "Ixz", "wing_loading_N_m2")} for k, c in CI.items()}).T.round(3))
display(fs.component_table()[["group", "mass [g]", "kind", "availability", "source"]])

# Part 5 — electronics and their energy

In [ ]:
display(save_table(fs.electrical_loads().round(2), "electrical_loads"))
display(fs.phase_power().round(1))
display(fs.servo_check(P).round(2))

# Part 6 — propulsion: calculate first, then select

The requirement at the altitudes that matter (the climb targets in EAS, TAS = EAS/√σ), then the selection: the
**T-Motor AT4125 KV540 on an APC 15x8E**, 6S — the only 41xx-class motor whose maker publishes a full-throttle point
with the rpm (21.36 V, 72.77 A, 8879 rpm, 5537 gf), which the motor model is fitted to; its 40 % point is the check.
The drive (`NisusPlusDrive`) is Boreas BEMT over airspeed × rpm, **signed**: at low rpm and speed the propeller windmills
(negative thrust and torque). At density ρ the propeller's forces are ×ρ/ρ0 at the same (V, rpm).

In [ ]:
AF0 = {"mass_kg": CI[fs.DEFAULT_PACK]["mass_kg"], "cd0": nisus_plus.drag_buildup(P, 20.0)["cd0"], "AR": L["AR"], "oswald": 0.9, "S": L["S_ref"]}
REQ = fs.requirements(AF0["mass_kg"], AF0["cd0"], AF0["AR"], AF0["oswald"], AF0["S"])
display(save_table(REQ.round(2), "requirements"))
display(fs.motor_shortlist_table(REQ.attrs["T_max_required_N"]))
MOTOR, FIT = fs.motor_model()
display(pd.Series({k: v for k, v in FIT.items() if not isinstance(v, (dict, str))}, name="motor fit").to_frame())
DR = fs.drive()
display(save_table(fs.drive_table(DR).round(2), "drive_vs_altitude"))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for h, col in ((0, "k"), (3000, "C0"), (4500, "C3")):
    rho = fs.atmosphere(h)["rho"]
    V = np.linspace(0, 35, 36)
    ax[0].plot(V, [DR.max_thrust(v, rho) for v in V], color=col, label=f"{h} m")
for V, col in ((15, "C2"), (25, "C1"), (30, "C4")):
    r = DR.rows(V)
    ax[1].plot(r["rpm"], r["thrust"], color=col, label=f"{V} m/s")
    ax[2].plot(r["rpm"], np.where(r["voltage"] > 0, r["voltage"] * r["current"], np.nan), color=col, label=f"{V} m/s")
    fw = DR.freewheel(V)
    ax[1].plot(fw["rpm"], fw["thrust"], "o", color=col)
ax[0].set(xlabel="airspeed [m/s]", ylabel="full-throttle thrust [N]", title="the drive against altitude"); ax[0].legend()
ax[1].axhline(0, color="k", lw=0.5); ax[1].set(xlabel="rpm", ylabel="thrust [N]", title="signed thrust (dots: the free windmill)", ylim=(-15, 40)); ax[1].legend()
ax[2].axhline(0, color="k", lw=0.5); ax[2].set(xlabel="rpm", ylabel="motor terminal power [W]", title="negative: generating", ylim=(-400, 800)); ax[2].legend()
for a in ax: a.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "plots" / "drive.png", dpi=110); plt.show()

## The Li-ion pack: sag, the cold, the charge window

`LiIonPack`: 21700 cells (Molicel P45B: 4.5 Ah, 45 A, 15 mΩ DC, charge ≤ 13.5 A, 70 g), 6S nP. The open-circuit voltage
curve, the resistance's growth in the cold and the capacity's loss are generic NMC assumptions; **no charging below
5 °C** (lithium plating): a cold pack gets nothing from the regeneration — the brake is then the short circuit.

In [ ]:
display(save_table(fs.pack_table().round(2), "packs"))
b = fs.pack()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
soc = np.linspace(0, 1, 101)
for T, col in ((25, "k"), (5, "C0"), (-10, "C3")):
    ax[0].plot(soc, b.terminal_v(70.0, soc, T), color=col, label=f"70 A, {T} °C")
ax[0].plot(soc, b.ocv(soc), "k--", label="open circuit"); ax[0].set(xlabel="state of charge", ylabel="pack voltage [V]"); ax[0].legend(); ax[0].grid(alpha=0.3)
I = np.linspace(0, 120, 61)
for T, col in ((25, "k"), (-10, "C3")):
    ax[1].plot(I, I * b.terminal_v(I, 0.5, T), color=col, label=f"{T} °C, half charge")
ax[1].set(xlabel="current [A]", ylabel="power [W]"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

# Part 7 — aerodynamics: the lattice with flaps, crow, the derivatives

PEREGRINE's vortex lattice on the wing split at the flap and aileron stations: a deflection is an incidence increment
τ K(δ) δ on its quads (plain-flap theory with the loss at large deflection: assumed). **Crow** — flaps 55° down,
ailerons 25° up — adds drag (Raymer's plain-flap formula: assumed until the crow CFD runs) at almost no change of lift
and a small pitching moment the elevator trims.

In [ ]:
A = ff.aero(P)
display(pd.Series({k: v for k, v in A.items() if isinstance(v, float)}, name="lattice").to_frame().round(4))
display(save_table(ff.crow_table(P, a=A).round(4), "crow_increments"))
CIZ = CI[fs.DEFAULT_PACK]
DER = ff.derivatives(P, x_cg_m=CIZ["x_cg_m"], z_cg_m=CIZ["z_cg_m"], a=A, h_m=3000.0)
display(save_table(DER, "derivatives"))
AF = {h: ff.airframe(fs.DEFAULT_PACK, P, h_m=h, a=A) for h in (0.0, 3000.0, 4500.0)}
display(pd.DataFrame({h: vars(a) for h, a in AF.items()}).T)

## Whole-aircraft CFD at altitude (Aeromant / OpenFOAM)

NISUS's RANS k-ω SST cases in the ISA air of 3000 m: the α sweep at the cruise EAS, **crow at V_FE** against the clean
aircraft at the same point (the check of the assumed ΔCD), the pusher as a rotor disk at its cruise rpm and **braked**.

In [ ]:
CRUISE_RPM = DR.at(ff.V_CRUISE_EAS / math.sqrt(fs.atmosphere(3000.0)["sigma"]), DR.throttle_for_thrust(20.9, 4.0, fs.atmosphere(3000.0)["rho"]), fs.atmosphere(3000.0)["rho"])["rpm"]
BRAKE_RPM = DR.brake(ff.V_FE_EAS / math.sqrt(fs.atmosphere(3000.0)["sigma"]), 1.0, fs.atmosphere(3000.0)["rho"])["rpm"]
if RUN_CFD:
    CFD_CASES = fcfd.plan(ROOT / "cfd", x_cg_m=CIZ["x_cg_m"], z_cg_m=CIZ["z_cg_m"], cruise_rpm=CRUISE_RPM, brake_rpm=BRAKE_RPM, fine=not QUICK)
    names = list(CFD_CASES)
    res = aeromant.run_cases([CFD_CASES[n] for n in names], jobs=1, processors=4, progress=True, cache=True)
    CFD = dict(zip(names, res))
else:
    CFD = {n: None for n in fcfd.specs(cruise_rpm=CRUISE_RPM, brake_rpm=BRAKE_RPM, fine=not QUICK)}
CFDT = fcfd.table(CFD, P)
STATUS["CFD"] = "run" if RUN_CFD and (CFDT["status"] == "solved").any() else "NOT RUN"
display(save_table(CFDT, "cfd_results"))
display(pd.Series(fcfd.crow_check(CFDT, DER), name="crow: CFD against the table").to_frame())

# Part 8 — structure

The load cases in **EAS** (the structure feels the dynamic pressure; Pratt's gust with the mass ratio at altitude), the
mountain gusts (10 m/s at V_C = 22 m/s EAS), crow at V_FE, the propeller brake's reverse thrust, a bungee launch and
the belly landing at 5.4 kg. The hand checks found the first spar (16/14 mm) and rear tube (12/10 mm) too weak for the
gust load factor and set 20/17 and 18/15; the first FEA found the 88 mm boom socket letting the boom pry its fitting
apart (now 170 mm) and the cradle failing as a beam (now a liner on the keel frame). The full run then **redesigned the
tail booms**: the 16/14 tubes held the tail + fin ultimate load (margin 0.33) but bent 50 mm at the tail — too soft for
pitch control and for flutter at altitude TAS; 20/18 tubes halve that for ~35 g. The heavier aircraft took the rear
carry-through tube to margin −0.04: it became 18/14 (a 2 mm wall; a 20 mm tube would have thinned the boom fitting's web
under its ring to margin −0.68 in the FEA). Two FEA lessons kept in `nisus_plus_structure`: the STEP files are keyed by the
parameters (a changed design once meshed a stale boom), and the boom's nominal stress leaves out the loaded sleeve. Talos models: the spar, the
**spar joiner** (new), the boom, the boom fitting, the motor mount in flight, braking and landing, the battery tray.

In [ ]:
LC = fst.load_cases(P)
display(save_table(LC, "load_cases"))
display(save_table(fst.wing_hand(P, LC), "wing_hand"))
display(save_table(fst.joints_hand(P, LC), "joints_hand"))

In [ ]:
FEA_CASES = fst.models(ROOT / "fea", P, LC)
FEA = {}
for c in FEA_CASES:
    FEA[c.name] = c.model.ensure(ROOT / "fea" / c.name, run=RUN_FEA, threads=4)
FEAT = pd.DataFrame([fst.summary_row(c, FEA[c.name]) for c in FEA_CASES]).set_index("case")
STATUS["FEA"] = "run" if (FEAT["status"] == "solved").any() else "NOT RUN"
display(save_table(FEAT.T, "fea_results"))
display(pd.DataFrame(fst.NOT_ESTABLISHED, columns=["not established", "why"]).set_index("not established"))

# Part 9 — flight against altitude

Trim (with crow), the CG range, the envelope at 0-6000 m (the stall rises in TAS, stays in EAS; the climb falls with the
density), the ceiling, **is a fast climb cheap?**, the **descent** in four configurations, **does regeneration pay?**,
the downdraft escape, the **launch** at the meadow and at a high site, the provisional limits.

In [ ]:
AFW = AF[3000.0]
display(pd.DataFrame([ff.trim(DER, AFW.mass_kg, v / math.sqrt(fs.atmosphere(h)["sigma"]), fs.atmosphere(h)["rho"]) | {"h": h} for h in (0, 3000, 4500) for v in (16, 18, 22)]).round(2))
display(pd.DataFrame([ff.trim_crow(DER, AFW.mass_kg, 22.0, RHO0, 1.0, g) | {"γ": g} for g in (0, 15, 30)]).round(2))
CGR = ff.cg_range(P, AFW.mass_kg, a=A)
display(pd.Series(CGR).to_frame("CG range"))
EL = fs.phase_power()["electronics battery-side [W]"]
ENV = ff.envelope_vs_altitude(AFW, DR, electronics_w=EL["survey"], battery_wh=fs.pack().energy_wh)
display(save_table(ENV.round(2), "envelope_vs_altitude"))
display(pd.Series(ff.ceiling(AFW, DR)).to_frame("ceiling"))
display(pd.Series(ff.ceiling(AFW, DR, dT=20.0)).to_frame("ceiling, ISA +20"))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for h, col in ((0, "k"), (1500, "C0"), (3000, "C2"), (4500, "C3"), (6000, "C4")):
    e = ff.envelope(AFW, DR, h)
    ax[0].plot(e["V"], e["roc"], color=col, label=f"{h} m")
    ax[1].plot(e["V"], e["P_level"], color=col, label=f"{h} m")
ax[0].axhline(5.0, color="r", ls=":", label="5 m/s target"); ax[0].set(xlabel="TAS [m/s]", ylabel="climb rate [m/s]", title="the climb against altitude")
ax[1].set(xlabel="TAS [m/s]", ylabel="level power, battery side [W]", ylim=(0, 600), title="level flight")
for a in ax: a.grid(alpha=0.3); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "plots" / "envelope.png", dpi=110); plt.show()
CS = ff.climb_strategy(DR, AFW)
display(save_table(CS.round(2), "climb_strategy"))

In [ ]:
DT = ff.descent_table(AFW, DR, DER, 3000.0)
display(save_table(DT.round(2), "descent_3000m"))
display(save_table(ff.best_descent(DT)[["EAS [m/s]", "TAS [m/s]", "sink [m/s]", "path angle [deg]", "battery side [W]"]].round(2), "best_descent"))
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, g in DT.groupby("configuration"):
    ax.plot(g["EAS [m/s]"], g["sink [m/s]"], "o-", label=name)
ax.axhline(10, color="r", ls=":", label="10 m/s target"); ax.axvline(ff.V_FE_EAS, color="k", ls="--", lw=0.8, label="V_FE (crow)")
ax.set(xlabel="EAS [m/s]", ylabel="sink rate [m/s]", title="the descent at 3000 m"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "plots" / "descent.png", dpi=110); plt.show()
REGEN = ff.regen_table(AFW, DR, DER)
display(save_table(REGEN.round(3), "regeneration"))
display(save_table(ff.downdraft_escape(AFW, DR).round(2), "downdraft_escape"))
display(save_table(ff.launch_table(AFW, DR, a=A, p=P).round(2), "launch"))
display(save_table(ff.operating_limits(AFW, DR, DER, CGR), "operating_limits"))

# Part 10 — mission energy over the mountains

The sizing mission: 300 s of ground operation at the meadow (1200 m), the climb to 4000 m step by step at each height's
density (6 m/s), 3 km to the survey area, 20 min of terrain-following survey, back, the crow + brake descent at 10 m/s
(its regeneration in its own row, from the descent table), the approach; 20 % reserve. The battery iteration compares
the packs.

In [ ]:
best = ff.best_descent(DT)
REGEN_W = -float(DT[(DT["configuration"] == "crow + propeller brake") & (DT["EAS [m/s]"] == 20.0)]["battery side [W]"].iloc[0])
PLAN = fs.MissionPlan(work_alt_m=4000.0, descent_regen_w=REGEN_W)
afd = ff.airframe_dict(AFW)
ME = fs.mission_energy(fs.pack(), DR, afd, PLAN)
display(save_table(ME["phases"].round(2), "mission_energy"))
display(pd.Series({k: ME[k] for k in ("E_nominal_wh", "E_available_wh", "E_used_wh", "E_left_pct_available", "E_reserve_wh", "reserve_kept", "trigger_wh",
                                      "t_climb_s", "E_climb_wh", "potential_energy_wh", "climb_efficiency", "t_survey_feasible_s")}).to_frame("mission"))
MTAB = fs.mission_table(DR, lambda k: ff.airframe_dict(ff.airframe(k, P, h_m=3000.0, a=A)), PLAN)
display(save_table(MTAB, "battery_iteration"))
COLD = fs.mission_energy(fs.pack(), DR, afd, replace(PLAN, pack_T_C=-5.0, dT=-10.0, descent_regen_w=0.0))
print(f"a cold morning (pack −5 °C, ISA −10): available {COLD['E_available_wh']:.0f} Wh, used {COLD['E_used_wh']:.0f} Wh, left {COLD['E_left_pct_available']:.0f} %, "
      f"reserve kept: {COLD['reserve_kept']}")

# Part 11 — the missions in MuJoCo over the mountains (Chiron)

The aircraft is one rigid body carrying every row of the mass table; `NisusPlusAero` (a subclass of NISUS's hook) applies
the forces of the derivative table with the crow increments, takes the **density from the ISA at the aircraft's
altitude**, runs the drive at that density and at the pack's **loaded voltage** (throttle, freewheel, or the brake that
**regenerates** within the pack's charge limit), integrates the battery-side energy. The terrain is the `Massif`, an
analytic valley (the meadow at 1200 m) between a north ridge (crest 3900-4300 m) and a lower south ridge, as a MuJoCo
height field of 10 x 9.5 km. `MountainWind` adds the log-law shear, the slope wind (lift on the windward face, the lee's
sink and turbulence) to NISUS's turbulence and gusts. The `NisusPlusController` holds the EAS with the pitch and the energy
with the throttle, then crow, then the brake; follows the terrain on the survey legs; spirals up and down over the
meadow; flies a 9° crow final along the valley.

The **launch** is a light bungee (17 m/s): the simulation found that a hand throw of 11-12 m/s leaves the 5.2 kg
aircraft at its stall at 1200 m and it sinks into the meadow (Part 9's launch check needs the take-off flap for it).

In [ ]:
lab = fsc.make_lab(fsc.Scenario(), log_geoms=False)
display(pd.Series(fr.inertia_check(lab, lab.robot.mass_table)).to_frame("MuJoCo vs the mass table"))
print("the hook's density at 3000 m:", lab.aero.air(3000.0)[0], "(ISA:", fs.atmosphere(3000.0)["rho"], ")")
M = fr.Massif()
X, Y = np.meshgrid(np.linspace(-3000, 7000, 300), np.linspace(-4000, 5500, 300))
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
cs = ax[0].contourf(X, Y, M.height(X, Y), levels=np.arange(1100, 4500, 200), cmap="terrain"); fig.colorbar(cs, ax=ax[0], label="m ASL")
scn0 = fsc.Scenario()
for y in scn0.survey_y:
    ax[0].plot(scn0.survey_x, [y, y], "r-", lw=2)
ax[0].plot(0, 0, "w*", ms=14); ax[0].set(title="the Massif: the meadow (star), the survey legs (red)", xlabel="x east [m]", ylabel="y north [m]"); ax[0].set_aspect("equal")
W = fr.MountainWind(steady=(0.0, 8.0, 0.0), massif=M); W.reset()
ys = np.linspace(-3500, 5000, 400)
ax[1].plot(ys, [W.vertical(2400.0, y, 150.0) for y in ys], label="south wind 8 m/s, 150 m above the ground")
W2 = fr.MountainWind(steady=(0.0, -10.0, 0.0), massif=M); W2.reset()
ax[1].plot(ys, [W2.vertical(2400.0, y, 150.0) for y in ys], label="north wind 10 m/s")
ax[1].axhline(0, color="k", lw=0.5); ax[1].set(xlabel="y north [m] (x = 2400)", ylabel="vertical air [m/s]", title="slope lift and lee sink"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "plots" / "massif.png", dpi=110); plt.show()

In [ ]:
SCNS = [fsc.Scenario()] if QUICK else fsc.standard_scenarios()
EPS = []
for scn in SCNS:
    t0 = time.time()
    labs = fsc.make_lab(scn, log_geoms=True)
    ep = fsc.run(labs, scn)
    EPS.append((scn, ep))
    fsc.timeseries(ep).to_csv(OUT / "telemetry" / f"{scn.slug}.csv", index=False)
    print(f"{scn.label:48s} {ep.outcome['reason']:20s} {ep.outcome['detail']}  ({time.time() - t0:.0f} s)")
    for e in ep.log["events"]:
        if e[1] in ("weather", "jetson", "energy", "companion lost", "landing", "descent") and "charged" not in e[2]:
            print(f"     {e[0]:7.1f} s  {e[1]}: {e[2]}")
CMP = fsc.compare([e for _, e in EPS])
display(save_table(CMP, "missions"))

In [ ]:
scn, ep = EPS[0]
display(save_table(fsc.phase_table(ep).round(2), f"phases_{scn.slug}"))
display(save_table(fsc.energy_table(ep).round(2), f"energy_{scn.slug}"))
fig, ax = plt.subplots(2, 2, figsize=(15, 8))
for scn_, ep_ in EPS:
    ts = fsc.timeseries(ep_)
    ax[0, 0].plot(ts["t"], ts["z_asl"], label=scn_.name)
    ax[0, 1].plot(ts["t"], ts["vz"], lw=0.7, label=scn_.name)
    ax[1, 0].plot(ts["t"], ts["E_used_Wh"], label=scn_.name)
    ax[1, 1].plot(ts["x"], ts["y"], lw=0.8, label=scn_.name)
ts = fsc.timeseries(EPS[0][1])
ax[0, 0].plot(ts["t"], ts["z_asl"] - ts["agl"], "k:", lw=0.8, label="the ground below (calm)")
ax[0, 0].set(ylabel="altitude [m]"); ax[0, 1].set(ylabel="climb rate [m/s]"); ax[1, 0].set(xlabel="t [s]", ylabel="energy used [Wh]")
ax[1, 1].contour(X, Y, M.height(X, Y), levels=np.arange(1500, 4400, 250), colors="0.75", linewidths=0.5); ax[1, 1].set_aspect("equal"); ax[1, 1].set(xlim=(-1500, 4000), ylim=(-1500, 3500), title="tracks")
for a in ax.flat: a.grid(alpha=0.3)
ax[0, 0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "plots" / "missions.png", dpi=110); plt.show()

In [ ]:
MOVIES = []
for scn, ep in (EPS[:1] if QUICK else EPS):
    path = OUT / f"{scn.slug}.mp4"
    try:
        fsc.render_movie(ep, scn, path, speed=30.0, fps=20, size=(640, 360) if QUICK else (960, 540))
        MOVIES.append(path); print("written", path)
    except Exception as e:
        print(f"{scn.label}: the movie needs pyvista off screen (xvfb-run): {type(e).__name__}: {e}")
if MOVIES:
    display(Video(str(MOVIES[0]), embed=False, width=800))

## Part 11b — chasing birds over the slopes

NISUS-Zero's bird-photography mission software (`vegeta.mission`, notebook 32: the detector, the multi-bird tracker,
the scheduler, stern photo passes with separation breakoffs, the shutter) on Nisus+ Zero, through NISUS's MuJoCo seam
(`nisus_birds.BirdMission`) unchanged. The birds are the mountains' (golden eagle, griffon vulture, alpine chough:
first estimates of their speeds and fear distances), beating along the face in its slope lift or circling wide in a
thermal, 150-450 m above the slope; the guidance works at NISUS+'s speeds and keeps 25 m from them (an eagle may attack a
drone); the terrain floor stays in force. YOLOX is not run (simulated detections at the ASSUMED Orin budget).

Two findings while setting it up. The tracker's range from the bird's size needs the **planned species' wingspan**:
with NISUS's 0.9 m prior a 2 m eagle's range came out half the truth and the shutter fired at 100-170 m (no good
photo). And a **tight thermal circle** (~90 m radius, about NISUS+'s own turn radius at 22 m/s TAS) cannot be
photographed with a fixed camera: the bird leaves the 34° frame before the range closes. Birds beating along a ridge
or circling wide can; the tight circle needs a gimbal (todo 5n.5). The search leg flies at the planned birds' height (a camera 15° down
sees small birds 50-100 m below only at ranges too long to detect them). The **alpine choughs** are photographed but
not well: a 0.8 m bird spans the good photo's 150 px only within ~28 m of the 6 mm lens, inside its flight distance —
an 8-12 mm lens (or a gimbal with a zoom) for small birds, the 6 mm for raptors.

In [ ]:
BSS = [fb.BirdScenario()] if QUICK else fb.bird_scenarios()
BEPS = []
for bs in BSS:
    t0 = time.time()
    ep = fb.run(bs, log_geoms=True)
    BEPS.append(ep)
    fb.photo_table(ep).to_csv(OUT / "tables" / f"{bs.slug}_photos.csv", index=False)
    print(f"{bs.name:34s} {ep.outcome['reason']:18s} ({time.time() - t0:.0f} s)")
BCMP = fb.compare(BEPS)
display(save_table(BCMP.T, "bird_hunts"))
display(fb.photo_table(BEPS[0]).head(15))

In [ ]:
import nisus_plus_birds_movie as fbm
BMOVIES = []
for ep in BEPS[:1]:
    path = OUT / f"{ep.bird_scenario.slug}.mp4"
    try:
        fbm.render_movie(ep, path)
        BMOVIES.append(path); print("written", path)
    except Exception as e:
        print(f"the bird video needs pyvista off screen (xvfb-run): {type(e).__name__}: {e}")
if BMOVIES:
    display(Video(str(BMOVIES[0]), embed=False, width=900))

# Part 12 — does regeneration pay? The first recommended build

**Does regeneration pay?** The numbers above (Part 9's regeneration table, Part 10's mission, Part 11's flights) say: in
energy, **no** — the braked propeller returns a few watt-hours per thousand metres of descent, about 2 % of the pack per
descent at best and nothing at all with a pack below 5 °C (the usual case after a long descent in mountain air). Its
value is the **drag**: the braked 15-inch propeller adds 5-15 N of drag at no cost in hardware (an AM32-class ESC with
active braking and a fixed propeller), and with crow it turns a 10 m/s descent into a 13-14 m/s one below V_FE — and
above V_FE, where crow must be retracted, the brake is the only airbrake left. So: keep the propeller brake **for the
drag**, count the regeneration as a bonus, never as reserve.

In [ ]:
BUILD = pd.Series({
    "airframe": "2.4 m twin-boom pusher, three-piece wing (XPS + 2 mm balsa D-box, 20/17 carbon spar, joiners), crow flaps and ailerons, H-tail on 16 mm booms",
    "mass (6S4P)": f"{CI['6s4p-p45b']['mass_kg']:.2f} kg, CG 28 % MAC, wing loading {CI['6s4p-p45b']['wing_loading_N_m2']:.0f} N/m²",
    "drive": "T-Motor AT4125 KV540 + APC 15x8E (fixed), 80 A 8S AM32-class ESC with active braking",
    "pack": f"6S4P Molicel P45B ({fs.pack().energy_wh:.0f} Wh, {fs.pack().mass_g:.0f} g) in an insulating sleeve",
    "computer": "Jetson Orin Nano Super on a 12 V buck, IMX477 + 6 mm, LTE link; the DEM of the area onboard",
    "avionics": "H743-WING class, pitot-static airspeed (EAS), lidar for the flare, ELRS 900 MHz, RFD900x",
    "launch / landing": "a light bungee (17 m/s) / a 9° crow approach to a belly landing on the skid",
    "climb (MTOW)": f"{ENV.loc['0 m', 'climb max [m/s]']:.1f} m/s at sea level, {ENV.loc['4500 m', 'climb max [m/s]']:.1f} m/s at 4500 m",
    "fast descent at 3000 m": f"{best.loc['crow + propeller brake', 'sink [m/s]']:.1f} m/s (crow + brake at {best.loc['crow + propeller brake', 'EAS [m/s]']:.0f} m/s EAS); "
                              f"brake alone at V_NE {best.loc['propeller brake', 'sink [m/s]']:.0f} m/s",
    "mountain mission (Part 10)": f"{ME['E_used_wh']:.0f} Wh of {ME['E_available_wh']:.0f} available, {ME['E_left_pct_available']:.0f} % left, survey up to {ME['t_survey_feasible_s'] / 60:.0f} min",
}, name="the first recommended build")
display(BUILD.to_frame())

## Assumptions to refine next

- **The brake region of the propeller** (Boreas drops the induced velocity at negative thrust; no turbulent-windmill
  state): measure the 15x8's drag and the regenerated current on a thrust stand in a wind tunnel or on a car.
- **Crow's drag and pitching moment** (Raymer's plain-flap formula, the lattice's linear K(δ)): the crow CFD here, then
  a flight test; buffet on the tail behind crow and behind the braked propeller.
- **The Li-ion model** (generic OCV, resistance and cold capacity; the 5 °C charge limit): cold-chamber tests of the pack.
- **The mountain wind** is a reduced model (slope wind, a lee factor): no rotors, no thermals, no valley winds.
- **Flutter at TAS**: V_NE is an EAS; at 4000 m the TAS is 25 % higher: Talos modes and a flutter estimate before the dives.
- **The launch**: the bungee's design; a hand launch with the take-off flap needs the flap channel in the controller.
- **The bird species** (speeds, fear distances) and their reactions to a 2.4 m aircraft: first estimates.
- **The terrain** is analytic and the controller knows it exactly: a real DEM (30 m) and its errors next.

In [ ]:
STATUS.update({"CAD": "run", "drawings": "run", "mass budgets": "run", "electronics": "run", "propulsion (Boreas, signed)": "run", "vortex lattice + crow": "run",
               "mission energy": "run", "MuJoCo missions": f"run ({len(EPS)} scenarios, {sum(e.outcome['success'] for _, e in EPS)} landed)",
               "bird hunts": f"run ({len(BEPS)})", "movies": f"{len(MOVIES) + len(BMOVIES)} written"})
if DEBUG:
    STATUS["FEA"] = STATUS["CFD"] = "MOCKED (VEGETA_DEBUG): the numbers mean nothing"
display(pd.Series(STATUS, name="status").to_frame())

In [ ]:
def plain(x):
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x
DESIGN = {"source": "33_nisus_plus", "params": {k: plain(v) for k, v in P.items()}, "layout": {k: plain(v) for k, v in L.items() if not isinstance(v, dict)},
          "mass": {k: {kk: plain(x) for kk, x in c.items() if kk != "inertia"} for k, c in CI.items()},
          "derivatives": {k: plain(x) for k, x in DER["value"].items()}, "motor_fit": {k: plain(x) for k, x in FIT.items() if not isinstance(x, (str, dict))},
          "envelope": {h: {k: plain(x) for k, x in row.items()} for h, row in ENV.T.to_dict().items()},
          "mission_energy": {k: plain(ME[k]) for k in ("E_used_wh", "E_left_pct_available", "reserve_kept", "t_survey_feasible_s", "trigger_wh")},
          "missions": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in CMP.T.to_dict().items()},
          "fea": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in FEAT.T.to_dict().items()},
          "cfd": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in CFDT.T.to_dict().items()}, "status": STATUS}
target = OUT / "nisus_plus_design.json" if DEBUG else Path("designs/data/nisus_plus_design.json")
target.write_text(json.dumps(DESIGN, indent=1, default=str))
print("written", target)